# RaMP Resolver Ambiguity Investigation

This notebook reproduces the RaMP ambiguous mapping analysis from the resolver parquet files.

It focuses on RaMP IDs in `chemicals/ambiguous/ramp.parquet` and classifies ambiguity using only InChIKey blocks first, then adds RaMP ClassyFire/ChemOnt metadata for the distinct-connectivity bucket.

Expected environment: run on dev2 from the `omnipath-build` virtual environment, or any environment with:

- `duckdb`
- `pandas`
- `pypath` with `pypath.inputs_v2.rampdb`
- resolver parquet under `DATA_ROOT`

In [ ]:
from __future__ import annotations

from pathlib import Path
from collections import Counter, defaultdict
import csv
import math
import re

import duckdb
import pandas as pd

# Adjust these when running outside dev2.
DATA_ROOT = Path('/home/omnipath/instances/dev2/data')
OUTPUT_DIR = Path('output')  # gitignored
OUTPUT_DIR.mkdir(exist_ok = True)

RAMP_AMBIGUOUS = DATA_ROOT / 'chemicals/ambiguous/ramp.parquet'
RAMP_LOOKUP = DATA_ROOT / 'chemicals/lookup/ramp.parquet'

assert RAMP_AMBIGUOUS.exists(), f'Missing {RAMP_AMBIGUOUS}'
assert RAMP_LOOKUP.exists(), f'Missing {RAMP_LOOKUP}'

con = duckdb.connect()
print(f'DATA_ROOT = {DATA_ROOT}')
print(f'OUTPUT_DIR = {OUTPUT_DIR.resolve()}')

## 1. Basic RaMP Resolver Counts

In [ ]:
ramp_counts = con.execute(
    f"""
    SELECT
        'unambiguous' AS table_name,
        count(*) AS row_count,
        count(DISTINCT key_value) AS ramp_id_count
    FROM read_parquet('{RAMP_LOOKUP}')
    UNION ALL
    SELECT
        'ambiguous' AS table_name,
        count(*) AS row_count,
        count(DISTINCT key_value) AS ramp_id_count
    FROM read_parquet('{RAMP_AMBIGUOUS}')
    """
).fetchdf()

ramp_counts.to_csv(OUTPUT_DIR / 'ramp_resolver_counts.tsv', sep = '\t', index = False)
ramp_counts

## 2. InChIKey-Only Ambiguity Classes

InChIKey blocks:

- first block: molecular connectivity
- second block: stereochemistry/isotope layer
- third block/final character: protonation/charge/final layer

The priority buckets are mutually exclusive. The flags table is non-exclusive.

In [ ]:
inchikey_class_sql = f"""
WITH parsed AS (
    SELECT
        key_value,
        canonical_identifier,
        split_part(canonical_identifier, '-', 1) AS connectivity,
        split_part(canonical_identifier, '-', 2) AS layer2,
        split_part(canonical_identifier, '-', 3) AS layer3,
        right(split_part(canonical_identifier, '-', 3), 1) AS protonation
    FROM read_parquet('{RAMP_AMBIGUOUS}')
), per_key AS (
    SELECT
        key_value,
        count(DISTINCT canonical_identifier) AS n_targets,
        count(DISTINCT connectivity) AS n_connectivity,
        count(DISTINCT layer2) AS n_layer2,
        count(DISTINCT connectivity || '-' || layer2) AS n_conn_layer2,
        count(DISTINCT layer3) AS n_layer3,
        count(DISTINCT protonation) AS n_protonation,
        bool_or(layer2 = 'UHFFFAOYSA') AS has_unspecified_layer2,
        count(DISTINCT CASE WHEN layer2 != 'UHFFFAOYSA' THEN layer2 END) AS n_specific_layer2
    FROM parsed
    GROUP BY key_value
), flags AS (
    SELECT *,
        n_connectivity > 1 AS distinct_connectivity,
        n_conn_layer2 > n_connectivity AS distinct_stereo_or_isotope_within_connectivity,
        n_layer3 > 1 AS distinct_final_layer,
        n_protonation > 1 AS distinct_protonation_charge,
        has_unspecified_layer2 AND n_specific_layer2 > 0 AS unspecified_vs_specific
    FROM per_key
), priority AS (
    SELECT *, CASE
        WHEN distinct_connectivity THEN 'distinct connectivity'
        WHEN distinct_stereo_or_isotope_within_connectivity AND unspecified_vs_specific THEN 'specified vs unspecified stereochemistry'
        WHEN distinct_stereo_or_isotope_within_connectivity THEN 'distinct stereochemistry/isotope layer'
        WHEN distinct_final_layer THEN 'distinct final/protonation layer'
        ELSE 'other'
    END AS priority_bucket
    FROM flags
)
"""

priority_buckets = con.execute(
    inchikey_class_sql + """
    SELECT
        priority_bucket,
        count(*) AS ramp_ids,
        sum(n_targets) AS mapping_rows
    FROM priority
    GROUP BY priority_bucket
    ORDER BY ramp_ids DESC
    """
).fetchdf()

nonexclusive_flags = con.execute(
    inchikey_class_sql + """
    SELECT 'TOTAL_AMBIGUOUS_RAMP_IDS' AS metric, count(*) AS ramp_ids FROM priority
    UNION ALL SELECT 'distinct_connectivity_flag', count(*) FROM priority WHERE distinct_connectivity
    UNION ALL SELECT 'distinct_stereo_or_isotope_within_connectivity_flag', count(*) FROM priority WHERE distinct_stereo_or_isotope_within_connectivity
    UNION ALL SELECT 'specified_vs_unspecified_layer2_flag', count(*) FROM priority WHERE unspecified_vs_specific
    UNION ALL SELECT 'distinct_final_layer_flag', count(*) FROM priority WHERE distinct_final_layer
    UNION ALL SELECT 'distinct_protonation_charge_flag', count(*) FROM priority WHERE distinct_protonation_charge
    ORDER BY metric
    """
).fetchdf()

priority_buckets.to_csv(OUTPUT_DIR / 'ramp_inchikey_priority_buckets.tsv', sep = '\t', index = False)
nonexclusive_flags.to_csv(OUTPUT_DIR / 'ramp_inchikey_nonexclusive_flags.tsv', sep = '\t', index = False)

priority_buckets

In [ ]:
nonexclusive_flags

## 3. Distinct-Connectivity Bucket Overview

This is the bucket most likely to contain true identity conflicts.

In [ ]:
distinct_connectivity_sql = inchikey_class_sql + """
, distinct_conn AS (
    SELECT * FROM priority WHERE distinct_connectivity
)
"""

distinct_connectivity_overview = con.execute(
    distinct_connectivity_sql + """
    SELECT
        count(*) AS ramp_ids,
        sum(n_targets) AS mapping_rows,
        min(n_targets) AS min_targets,
        median(n_targets) AS median_targets,
        max(n_targets) AS max_targets,
        min(n_connectivity) AS min_connectivities,
        median(n_connectivity) AS median_connectivities,
        max(n_connectivity) AS max_connectivities,
        count(*) FILTER (WHERE n_targets = n_connectivity) AS one_target_per_connectivity,
        count(*) FILTER (WHERE n_targets > n_connectivity) AS multiple_targets_within_connectivity
    FROM distinct_conn
    """
).fetchdf()

distinct_connectivity_by_connectivity_count = con.execute(
    distinct_connectivity_sql + """
    SELECT
        n_connectivity,
        count(*) AS ramp_ids,
        sum(n_targets) AS mapping_rows
    FROM distinct_conn
    GROUP BY n_connectivity
    ORDER BY n_connectivity
    """
).fetchdf()

distinct_connectivity_by_target_count = con.execute(
    distinct_connectivity_sql + """
    SELECT
        n_targets,
        count(*) AS ramp_ids,
        sum(n_connectivity) AS connectivity_instances
    FROM distinct_conn
    GROUP BY n_targets
    ORDER BY n_targets
    """
).fetchdf()

for name, frame in {
    'distinct_connectivity_overview': distinct_connectivity_overview,
    'distinct_connectivity_by_connectivity_count': distinct_connectivity_by_connectivity_count,
    'distinct_connectivity_by_target_count': distinct_connectivity_by_target_count,
}.items():
    frame.to_csv(OUTPUT_DIR / f'ramp_{name}.tsv', sep = '\t', index = False)

distinct_connectivity_overview

In [ ]:
distinct_connectivity_by_connectivity_count

In [ ]:
distinct_connectivity_by_target_count

## 4. Load RaMP Metadata For Distinct-Connectivity Rows

This joins each ambiguous InChIKey back to RaMP `chem_props`, then to RaMP `metabolite_class` rows.

In [ ]:
distinct_rows = con.execute(
    f"""
    WITH parsed AS (
        SELECT
            key_value AS ramp_id,
            canonical_identifier AS inchikey,
            split_part(canonical_identifier, '-', 1) AS connectivity
        FROM read_parquet('{RAMP_AMBIGUOUS}')
    ), per_key AS (
        SELECT ramp_id, count(DISTINCT connectivity) AS n_connectivity
        FROM parsed
        GROUP BY ramp_id
    )
    SELECT p.ramp_id, p.inchikey, p.connectivity
    FROM parsed p
    JOIN per_key k USING (ramp_id)
    WHERE k.n_connectivity > 1
    ORDER BY p.ramp_id, p.inchikey
    """
).fetchdf()

distinct_ramp_ids = set(distinct_rows['ramp_id'])
ik_to_conn = {
    (row.ramp_id, row.inchikey): row.connectivity
    for row in distinct_rows.itertuples(index = False)
}
conn_by_ramp = defaultdict(set)
inchikeys_by_ramp = defaultdict(set)
for row in distinct_rows.itertuples(index = False):
    conn_by_ramp[row.ramp_id].add(row.connectivity)
    inchikeys_by_ramp[row.ramp_id].add(row.inchikey)

from pypath.inputs_v2.rampdb import resource as ramp_resource

source_to_conn = {}
meta_by_conn = defaultdict(lambda: defaultdict(lambda: {
    'names': set(),
    'formulas': set(),
    'sources': set(),
    'source_ids': set(),
    'masses': [],
}))

for raw in ramp_resource.chem_props.raw():
    ramp_id = raw.get('ramp_id')
    inchikey = raw.get('inchi_key')
    key = (ramp_id, inchikey)
    if key not in ik_to_conn:
        continue

    conn = ik_to_conn[key]
    source_id = str(raw.get('chem_source_id') or '').strip().lower()
    if source_id:
        source_to_conn[(ramp_id, source_id)] = conn

    meta = meta_by_conn[ramp_id][conn]
    for raw_key, bucket in (
        ('common_name', 'names'),
        ('mol_formula', 'formulas'),
        ('chem_data_source', 'sources'),
        ('chem_source_id', 'source_ids'),
    ):
        value = str(raw.get(raw_key) or '').strip()
        if value:
            meta[bucket].add(value)

    mass = raw.get('monoisotop_mass')
    if isinstance(mass, (int, float)) and not math.isnan(mass):
        meta['masses'].append(float(mass))

classes_by_conn = defaultdict(lambda: defaultdict(lambda: defaultdict(set)))
for raw in ramp_resource.metabolite_class.raw():
    ramp_id = raw.get('ramp_id')
    if ramp_id not in distinct_ramp_ids:
        continue

    source_id = str(raw.get('class_source_id') or '').strip().lower()
    conn = source_to_conn.get((ramp_id, source_id))
    if conn is None:
        continue

    level = raw.get('class_level_name')
    label = str(raw.get('class_name') or '').strip()
    if level and label:
        classes_by_conn[ramp_id][conn][level].add(label)

print(f'distinct connectivity RaMP IDs: {len(distinct_ramp_ids)}')
print(f'distinct connectivity target rows: {len(distinct_rows)}')
print(f'RaMP IDs with any ClassyFire labels: {sum(bool(classes_by_conn.get(r)) for r in distinct_ramp_ids)}')

## 5. ClassyFire/ChemOnt Buckets For Distinct-Connectivity Rows

Bucket rule:

1. shared subclass -> low concern
2. shared class only -> medium
3. shared superclass only -> medium
4. no shared superclass -> high
5. missing/partial labels -> unknown

In [ ]:
def shared_labels(ramp_id: str, level: str) -> set[str] | None:
    label_sets = []
    for conn in conn_by_ramp[ramp_id]:
        labels = classes_by_conn[ramp_id][conn].get(level, set())
        if not labels:
            return None
        label_sets.append(labels)
    return set.intersection(*label_sets) if label_sets else set()


def classify_classyfire(ramp_id: str) -> str:
    if not classes_by_conn.get(ramp_id):
        return 'Unknown - missing classification'

    if any(not classes_by_conn[ramp_id].get(conn) for conn in conn_by_ramp[ramp_id]):
        return 'Unknown - partial classification'

    for level, label in (
        ('ClassyFire_sub_class', 'Low - shared subclass'),
        ('ClassyFire_class', 'Medium - shared class only'),
        ('ClassyFire_super_class', 'Medium - shared superclass only'),
    ):
        shared = shared_labels(ramp_id, level)
        if shared:
            return label
        if shared is None:
            return 'Unknown - partial classification'

    return 'High - no shared superclass'


def labels_by_conn(ramp_id: str, level: str) -> str:
    parts = []
    for conn in sorted(conn_by_ramp[ramp_id]):
        labels = sorted(classes_by_conn[ramp_id][conn].get(level, []))
        parts.append('/'.join(labels) if labels else '[missing]')
    return ' | '.join(parts)


def names_by_conn(ramp_id: str, limit: int = 4) -> str:
    parts = []
    for conn in sorted(conn_by_ramp[ramp_id]):
        names = sorted(meta_by_conn[ramp_id][conn]['names'])[:limit]
        parts.append('/'.join(names) if names else '[missing]')
    return ' | '.join(parts)


def formulas_by_conn(ramp_id: str) -> str:
    parts = []
    for conn in sorted(conn_by_ramp[ramp_id]):
        formulas = sorted(meta_by_conn[ramp_id][conn]['formulas'])
        parts.append('/'.join(formulas) if formulas else '[missing]')
    return ' | '.join(parts)


def source_sets_by_conn(ramp_id: str) -> str:
    parts = []
    for conn in sorted(conn_by_ramp[ramp_id]):
        sources = sorted(meta_by_conn[ramp_id][conn]['sources'])
        parts.append('/'.join(sources) if sources else '[missing]')
    return ' | '.join(parts)


def reason_for_bucket(ramp_id: str, bucket: str) -> str:
    if bucket == 'Medium - shared class only':
        shared = ', '.join(sorted(shared_labels(ramp_id, 'ClassyFire_class') or []))
        return (
            f'shared ClassyFire class: {shared}; '
            f'subclasses: {labels_by_conn(ramp_id, "ClassyFire_sub_class")}; '
            f'names: {names_by_conn(ramp_id)}'
        )
    if bucket == 'Medium - shared superclass only':
        shared = ', '.join(sorted(shared_labels(ramp_id, 'ClassyFire_super_class') or []))
        return (
            f'shared ClassyFire superclass: {shared}; '
            f'classes: {labels_by_conn(ramp_id, "ClassyFire_class")}; '
            f'names: {names_by_conn(ramp_id)}'
        )
    if bucket == 'High - no shared superclass':
        return (
            f'superclasses: {labels_by_conn(ramp_id, "ClassyFire_super_class")}; '
            f'classes: {labels_by_conn(ramp_id, "ClassyFire_class")}; '
            f'names: {names_by_conn(ramp_id)}'
        )
    return ''

classyfire_rows = []
for ramp_id in sorted(distinct_ramp_ids):
    bucket = classify_classyfire(ramp_id)
    classyfire_rows.append({
        'RaMP ID': ramp_id,
        'ClassyFire Bucket': bucket,
        'Target Count': len(inchikeys_by_ramp[ramp_id]),
        'Connectivity Count': len(conn_by_ramp[ramp_id]),
        'InChIKeys': '; '.join(sorted(inchikeys_by_ramp[ramp_id])),
        'Sources By Connectivity': source_sets_by_conn(ramp_id),
        'Formulas By Connectivity': formulas_by_conn(ramp_id),
        'Superclasses By Connectivity': labels_by_conn(ramp_id, 'ClassyFire_super_class'),
        'Classes By Connectivity': labels_by_conn(ramp_id, 'ClassyFire_class'),
        'Subclasses By Connectivity': labels_by_conn(ramp_id, 'ClassyFire_sub_class'),
        'Names By Connectivity': names_by_conn(ramp_id),
    })

classyfire_detail = pd.DataFrame(classyfire_rows)
classyfire_summary = (
    classyfire_detail
    .groupby('ClassyFire Bucket', as_index = False)
    .agg(
        ramp_ids = ('RaMP ID', 'count'),
        mapping_rows = ('Target Count', 'sum'),
    )
    .sort_values('ramp_ids', ascending = False)
)

classyfire_detail.to_csv(OUTPUT_DIR / 'ramp_distinct_connectivity_classyfire_detail.tsv', sep = '\t', index = False)
classyfire_summary.to_csv(OUTPUT_DIR / 'ramp_distinct_connectivity_classyfire_summary.tsv', sep = '\t', index = False)
classyfire_summary

## 6. Medium And High Examples Table

This is the table used for message-ready examples, including all ambiguous InChIKeys per RaMP ID.

In [ ]:
medium_high_examples = classyfire_detail[
    classyfire_detail['ClassyFire Bucket'].isin([
        'Medium - shared class only',
        'Medium - shared superclass only',
        'High - no shared superclass',
    ])
].copy()
medium_high_examples['Reason'] = medium_high_examples.apply(
    lambda row: reason_for_bucket(row['RaMP ID'], row['ClassyFire Bucket']),
    axis = 1,
)
medium_high_examples = medium_high_examples.rename(columns = {'ClassyFire Bucket': 'Priority'})[
    ['Priority', 'RaMP ID', 'InChIKeys', 'Reason']
]

medium_high_examples.to_csv(OUTPUT_DIR / 'ramp_medium_high_examples.tsv', sep = '\t', index = False)
medium_high_examples

## 7. Optional ChEBI Ancestry Analysis

This section is slower because it parses ChEBI. It is useful as a secondary signal, but coverage is sparse for this RaMP bucket.

In [ ]:
RUN_CHEBI_ANCESTRY = False

In [ ]:
if RUN_CHEBI_ANCESTRY:
    from pypath.inputs_v2.chebi import resource as chebi_resource

    chebi_by_conn = defaultdict(lambda: defaultdict(set))
    for raw in ramp_resource.chem_props.raw():
        ramp_id = raw.get('ramp_id')
        inchikey = raw.get('inchi_key')
        key = (ramp_id, inchikey)
        if key not in ik_to_conn:
            continue

        source = str(raw.get('chem_data_source') or '').strip().lower()
        source_id = str(raw.get('chem_source_id') or '').strip()
        if source != 'chebi' and not source_id.lower().startswith('chebi:'):
            continue

        match = re.search(r'(?:CHEBI:)?(\d+)$', source_id, re.I)
        if match:
            chebi_by_conn[ramp_id][ik_to_conn[key]].add(f'CHEBI:{match.group(1)}')

    needed_chebi = {
        chebi_id
        for ramp_id in distinct_ramp_ids
        for conn in conn_by_ramp[ramp_id]
        for chebi_id in chebi_by_conn[ramp_id].get(conn, set())
    }

    chebi_info = {}
    ancestor_ids = set()
    for raw in chebi_resource.molecules.raw():
        chebi_id = raw.get('chebi_id')
        if chebi_id in needed_chebi:
            ancestors = set(raw.get('ancestor_terms') or [])
            chebi_info[chebi_id] = {
                'name': raw.get('name') or '',
                'ancestors': ancestors,
            }
            ancestor_ids.update(ancestors)

    all_chebi_ids = needed_chebi | ancestor_ids
    for raw in chebi_resource.molecules.raw():
        chebi_id = raw.get('chebi_id')
        if chebi_id in all_chebi_ids and chebi_id not in chebi_info:
            chebi_info[chebi_id] = {
                'name': raw.get('name') or '',
                'ancestors': set(raw.get('ancestor_terms') or []),
            }

    def chebi_terms(ids: set[str]) -> set[str]:
        result = set()
        for chebi_id in ids:
            info = chebi_info.get(chebi_id)
            if info:
                result.add(chebi_id)
                result.update(info['ancestors'])
        return result

    def chebi_depth(term: str) -> int:
        return len(chebi_info.get(term, {}).get('ancestors', set()))

    def useful_chebi_ancestor(term: str) -> bool:
        name = chebi_info.get(term, {}).get('name', '').lower()
        if chebi_depth(term) < 10:
            return False
        generic_fragments = [
            'molecular entity', 'chemical entity', 'elemental ', ' atom',
            'subatomic', 'particle', 'role', 'application', 'group molecular',
            'p-block', 's-block', 'd-block', 'main group', 'chalcogen',
            'pnictogen', 'cation', 'anion',
        ]
        return bool(name) and not any(fragment in name for fragment in generic_fragments)

    chebi_rows = []
    for ramp_id in sorted(distinct_ramp_ids):
        per_conn_terms = []
        any_chebi = False
        all_conn_have_chebi = True
        for conn in conn_by_ramp[ramp_id]:
            ids = chebi_by_conn[ramp_id].get(conn, set())
            any_chebi = any_chebi or bool(ids)
            all_conn_have_chebi = all_conn_have_chebi and bool(ids)
            per_conn_terms.append(chebi_terms(ids))

        best_term = None
        if not any_chebi:
            bucket = 'no_chebi_coverage'
        elif not all_conn_have_chebi:
            bucket = 'partial_chebi_coverage'
        else:
            common = set.intersection(*per_conn_terms) if per_conn_terms else set()
            useful = {term for term in common if useful_chebi_ancestor(term)}
            if useful:
                best_term = max(useful, key = lambda term: (chebi_depth(term), chebi_info[term]['name']))
                bucket = 'shared_useful_chebi_ancestor'
            elif common:
                best_term = max(common, key = lambda term: (chebi_depth(term), chebi_info.get(term, {}).get('name', '')))
                bucket = 'only_generic_shared_chebi_ancestor'
            else:
                bucket = 'no_shared_chebi_ancestor'

        chebi_rows.append({
            'RaMP ID': ramp_id,
            'ChEBI Bucket': bucket,
            'Best Shared ChEBI Ancestor': best_term or '',
            'Best Shared ChEBI Ancestor Name': chebi_info.get(best_term, {}).get('name', '') if best_term else '',
            'Best Shared ChEBI Ancestor Depth': chebi_depth(best_term) if best_term else '',
        })

    chebi_detail = pd.DataFrame(chebi_rows)
    chebi_summary = chebi_detail.groupby('ChEBI Bucket', as_index = False).agg(ramp_ids = ('RaMP ID', 'count'))
    chebi_detail.to_csv(OUTPUT_DIR / 'ramp_distinct_connectivity_chebi_detail.tsv', sep = '\t', index = False)
    chebi_summary.to_csv(OUTPUT_DIR / 'ramp_distinct_connectivity_chebi_summary.tsv', sep = '\t', index = False)
    display(chebi_summary)
else:
    print('Set RUN_CHEBI_ANCESTRY = True to run the optional ChEBI analysis.')